# IR → Lean: дообучение перевода (self-contained, для Kaggle)

Самодостаточный ноутбук: нужен только `dataset.json`
(пары `Haskell-IR → Lean-код`).

Ноутбук сам найдёт `dataset.json` в `/kaggle/input/...`. Модель учится переводить общий IR
в код Lean; held-out split показывает генерализацию.

In [1]:
import json, os, glob, difflib, random
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL    = "t5-small"
PREFIX   = "translate IR to Lean: "
VAL_FRAC = 0.2
EPOCHS   = 200
LR       = 5e-4
SEED     = 0
torch.manual_seed(SEED); random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [2]:
DATA = "/kaggle/input/datasets/liderpartiii/research-ir-dataset/dataset.json"
rows = json.loads(open(DATA).read())
print(f"dataset: {DATA}   ({len(rows)} пар)")
print("пример:", rows[0]["name"])

dataset: /kaggle/input/datasets/liderpartiii/research-ir-dataset/dataset.json   (69 пар)
пример: idB


In [3]:
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(rows), generator=g).tolist()
n_val = max(1, int(len(rows) * VAL_FRAC))
val_idx = set(perm[:n_val])
train_rows = [r for i, r in enumerate(rows) if i not in val_idx]
val_rows   = [r for i, r in enumerate(rows) if i in val_idx]
print(f"train={len(train_rows)}  val={len(val_rows)}")

train=56  val=13


In [4]:
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device)

enc = tok([PREFIX + r["haskell_ir"] for r in train_rows],
          padding=True, truncation=True, max_length=512, return_tensors="pt")
labels = tok(text_target=[r["lean_src"] for r in train_rows],
             padding=True, truncation=True, max_length=256, return_tensors="pt").input_ids
labels[labels == tok.pad_token_id] = -100        # игнор pad в loss
enc = {k: v.to(device) for k, v in enc.items()}
labels = labels.to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
opt = torch.optim.AdamW(model.parameters(), lr=LR)
model.train()
for epoch in range(1, EPOCHS + 1):
    opt.zero_grad()
    loss = model(**enc, labels=labels).loss
    loss.backward()
    opt.step()
    if epoch % 20 == 0 or epoch == 1:
        print(f"epoch {epoch:4d}   loss {loss.item():.4f}")

epoch    1   loss 4.4780
epoch   20   loss 0.2915
epoch   40   loss 0.0694
epoch   60   loss 0.0260
epoch   80   loss 0.0179
epoch  100   loss 0.0144
epoch  120   loss 0.0101
epoch  140   loss 0.0107
epoch  160   loss 0.0109
epoch  180   loss 0.0072
epoch  200   loss 0.0043


In [6]:
def evaluate(rows):
    model.eval()
    exact, sim_sum, shown = 0, 0.0, []
    for r in rows:
        ids = tok(PREFIX + r["haskell_ir"], return_tensors="pt",
                  truncation=True, max_length=512).input_ids.to(device)
        with torch.no_grad():
            gen = model.generate(ids, max_length=200, num_beams=4)
        pred = tok.decode(gen[0], skip_special_tokens=True).strip()
        gold = r["lean_src"].strip()
        ok = pred == gold
        exact += ok
        sim_sum += difflib.SequenceMatcher(None, pred, gold).ratio()
        shown.append((r["name"], ok, gold, pred))
    return exact, sim_sum / max(1, len(rows)), shown

tr_e, tr_s, _ = evaluate(train_rows)
val_e, val_s, shown = evaluate(val_rows)

print("\nHeld-out (модель не видела эти функции при обучении)")
for name, ok, gold, pred in shown:
    print(f"\n[{name}]  {'OK' if ok else 'x'}")
    print("  gold:", gold)
    print("  pred:", pred)

print(f"\ntrain:  exact {tr_e}/{len(train_rows)}   близость {tr_s:.1%}")
print(f"val:    exact {val_e}/{len(val_rows)}   близость {val_s:.1%}")


Held-out (модель не видела эти функции при обучении)

[constFalse]  OK
  gold: def constFalse (b : Bool) : Bool := false
  pred: def constFalse (b : Bool) : Bool := false

[andB]  OK
  gold: def andB (a b : Bool) : Bool := match a with | true => b | false => false
  pred: def andB (a b : Bool) : Bool := match a with | true => b | false => false

[orB]  OK
  gold: def orB (a b : Bool) : Bool := match a with | true => true | false => b
  pred: def orB (a b : Bool) : Bool := match a with | true => true | false => b

[nimplyB]  OK
  gold: def nimplyB (a b : Bool) : Bool := andB a (notB b)
  pred: def nimplyB (a b : Bool) : Bool := andB a (notB b)

[and3]  OK
  gold: def and3 (a b c : Bool) : Bool := andB a (andB b c)
  pred: def and3 (a b c : Bool) : Bool := andB a (andB b c)

[majority3]  x
  gold: def majority3 (a b c : Bool) : Bool := orB (andB a b) (orB (andB a c) (andB b c))
  pred: def majority3 (a b c : Bool) : Bool := orB (andB a b) (orB (andB a c) (andB b c)

[addMul]  OK
  gold: